# Understanding the kmeans2 file

In [12]:
import os
import argparse
from typing import Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.colors
import matplotlib.patches as mpatches
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from skimage.transform import resize
import sunpy.map

matplotlib.use('Agg')  # Use Agg backend for saving files



## Utilities  

In [ ]:

# --- Helper Functions ---

def load_fits_data(channel_dir: str) -> Tuple[np.ndarray, dict]:
    """
    Load FITS data and metadata from a single channel directory.
    Loads only the *first* FITS file in the directory.

    Args:
        channel_dir (str): Path to the channel directory.

    Returns:
        Tuple[np.ndarray, dict]: Data and metadata from the FITS file.

    Raises:
        FileNotFoundError: If no FITS files are found in the directory.
    """
    fits_files = [
        f for f in os.listdir(channel_dir) if f.endswith(".fits")
    ]
    if not fits_files:
        raise FileNotFoundError(
            f"No FITS files found in directory: {channel_dir}"
        )

    fits_path = os.path.join(
        channel_dir, fits_files[0]
    )  # Load the first FITS file
    aia_map = sunpy.map.Map(fits_path)
    return aia_map.data, aia_map.meta






def create_circular_mask(data: np.ndarray, metadata: dict) -> np.ndarray:
    """
    Creates a circular mask for the solar disk based on metadata.

    Args:
        data (np.ndarray): Image data.
        metadata (dict): FITS metadata containing header info.

    Returns:
        np.ndarray: Boolean mask, True for pixels inside the solar disk.
    """
    ny, nx = data.shape
    x_center, y_center = nx // 2, ny // 2
    cdelt1 = metadata.get("cdelt1", 1.0)  # Arcsec/pixel in X
    solar_radius_arcsec = metadata.get("rsun_obs", 960.0)  # Solar radius arcsec
    solar_radius_pixels = int(solar_radius_arcsec / abs(cdelt1))

    y, x = np.ogrid[:ny, :nx]
    distance_from_center = np.sqrt((x - x_center)**2 + (y - y_center)**2)
    return distance_from_center <= solar_radius_pixels


def preprocess_image(
    data: np.ndarray, mask: np.ndarray, size: int = 512
) -> np.ndarray:
    """
    Resizes the image and applies the mask, setting masked areas to NaN.

    Args:
        data (np.ndarray): Input image data.
        mask (np.ndarray): Boolean mask for the solar disk.
        size (int): Desired size of the resized image (default: 512).

    Returns:
        np.ndarray: Preprocessed image data with mask applied and resized.
    """
    resized_data = resize(
        data, (size, size), mode='reflect', anti_aliasing=True
    )
    resized_mask = resize(
        mask, (size, size), mode='reflect', anti_aliasing=False
    ) > 0.5
    masked_data = resized_data.copy()
    masked_data[~resized_mask] = np.nan  # Apply mask
    return masked_data



## Data preparation

In [ ]:

# --- Data Preparation and Anomaly Detection ---

def prepare_data_concatenated(masked_data_list: list) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Concatenates masked data from multiple channels, handles NaNs,
    and scales the data.

    Args:
        masked_data_list (list): List of masked data arrays (one per channel).

    Returns:
        Tuple[np.ndarray, np.ndarray, np.ndarray]:
            - scaled_data: Scaled data, reshaped to
              (pixels_without_nan, num_channels).
            - valid_pixel_mask: 1D boolean mask (True for valid pixels).
            - nan_mask: 1D boolean mask (True for pixels with NaN in any channel).
    """
    stacked_data = np.stack(
        masked_data_list, axis=-1
    )  # (height, width, channels)
    reshaped_data = stacked_data.reshape(
        (-1, len(masked_data_list))
    )  # (pixels, channels)
    nan_mask = np.isnan(reshaped_data).any(axis=1)  # Rows with any NaN
    cleaned_data = reshaped_data[~nan_mask]  # Remove rows with NaNs
    scaler = RobustScaler()
    scaled_data = scaler.fit_transform(cleaned_data)
    return scaled_data, ~nan_mask, nan_mask


def detect_anomalies_isolation_forest(
    data: np.ndarray, contamination: float
) -> np.ndarray:
    """
    Detects anomalies using Isolation Forest and returns anomaly scores.

    Args:
        data (np.ndarray): Input data for anomaly detection.
        contamination (float): Expected proportion of anomalies in the data.

    Returns:
        np.ndarray: Anomaly scores for each data point.
                     Lower scores indicate higher anomaly.
    """
    iso_forest = IsolationForest(
        contamination=contamination, random_state=42
    )
    iso_forest.fit(data)
    return iso_forest.decision_function(data)  # Return anomaly *scores*



## Clustering

In [ ]:

# --- Clustering ---

def perform_kmeans_clustering(
    data: np.ndarray, n_clusters: int, random_state: int = 42
) -> Tuple[np.ndarray, float]:
    """
    Performs K-Means clustering.

    Args:
        data (np.ndarray): Data to be clustered.
        n_clusters (int): Number of clusters.
        random_state (int): Random seed for reproducibility (default: 42).

    Returns:
        Tuple[np.ndarray, float]:
            - labels: Cluster labels for each data point.
            - inertia: Sum of squared distances to closest centroid.
    """
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )  # Use n_init for multiple initializations
    kmeans.fit(data)
    return kmeans.labels_, kmeans.inertia_


def determine_optimal_k_elbow(
    data: np.ndarray, max_k: int = 10, random_state: int = 42
) -> int:
    """
    Applies the Elbow Method to suggest an optimal number of clusters (k).

    Args:
        data (np.ndarray): Data to be clustered.
        max_k (int): Maximum number of clusters to test (default: 10).
        random_state (int): Random seed for reproducibility (default: 42).

    Returns:
        int: Suggested optimal number of clusters (k).
    """
    inertias = []
    for k in range(1, max_k + 1):
        _, inertia = perform_kmeans_clustering(data, k, random_state)
        inertias.append(inertia)

    # Plot the Elbow curve
    plt.figure(figsize=(8, 6))
    plt.plot(range(1, max_k + 1), inertias, marker='o')
    plt.title('Elbow Method For Optimal k')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Inertia')
    plt.grid(True)
    elbow_plot_path = "elbow_plot.png"
    plt.savefig(elbow_plot_path)
    plt.close()
    print(f"Elbow plot saved to: {elbow_plot_path}")

    # Find the "elbow" (simplistic approach - could be improved)
    diffs = np.diff(inertias)
    diffs2 = np.diff(diffs)
    optimal_k = np.argmax(diffs2) + 2  # +2 because of two diff operations

    return optimal_k


def create_cluster_mask(
    anomaly_mask: np.ndarray,
    labels: np.ndarray,
    valid_pixel_mask: np.ndarray,
    image_size: int
) -> Tuple[np.ndarray, matplotlib.colors.ListedColormap, list, int]:
    """
    Creates a 2D cluster mask from anomaly mask and cluster labels.

    Args:
        anomaly_mask (np.ndarray): 2D boolean mask of anomaly pixels.
        labels (np.ndarray): Cluster labels for each anomaly pixel.
        valid_pixel_mask (np.ndarray): 1D boolean mask of valid pixels.
        image_size (int): Size of the image (height/width).

    Returns:
        Tuple[np.ndarray, matplotlib.colors.ListedColormap, list, int]:
            - cluster_mask: 2D array representing cluster assignments.
            - cluster_cmap: Colormap used for clusters.
            - cluster_patches: Legend patches for the clusters.
            - n_clusters: Number of clusters.
    """
    print("-" * 20)
    print("Inside create_cluster_mask:")
    print(f"anomaly_mask.shape: {anomaly_mask.shape}")
    print(f"np.sum(anomaly_mask): {np.sum(anomaly_mask)}")
    # Number of initially detected anomalies

    cluster_mask = np.zeros_like(anomaly_mask, dtype=int)  # Initialize zeros
    n_clusters = 0
    cluster_cmap = matplotlib.colors.ListedColormap([])  # Initialize empty cmap
    cluster_patches = []

    if len(labels) > 0:
        n_clusters = len(np.unique(labels))
        print(f"Number of clusters (n_clusters): {n_clusters}")
        # Modified color palette, removed dark blue, added more distinct colors
        cluster_colors = [
            '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
            '#8c564b', '#e377c2', '#7f7f7f'  # No blue, added grey
        ]
        cluster_cmap = matplotlib.colors.ListedColormap(
            cluster_colors[:n_clusters]
        )

        # Create 2D masks for easier indexing
        valid_pixel_mask_2d = valid_pixel_mask.reshape((image_size, image_size))
        anomaly_pixels_indices = np.argwhere(anomaly_mask)  # 2D anomaly indices
        print(f"len(anomaly_pixels_indices): {len(anomaly_pixels_indices)}")
        # Should be same as np.sum(anomaly_mask)

        # Map 2D anomaly pixel indices to rows in prepared data
        valid_pixel_indices_2d = np.argwhere(valid_pixel_mask_2d)
        pixel_index_map = {
            tuple(index_2d): i for i, index_2d in enumerate(valid_pixel_indices_2d)
        }

        valid_anomaly_pixel_indices = []
        for anomaly_pixel_index_2d in anomaly_pixels_indices:
            if tuple(anomaly_pixel_index_2d) in pixel_index_map:  # Check valid
                valid_anomaly_pixel_indices.append(anomaly_pixel_index_2d)
        valid_anomaly_pixel_indices = np.array(valid_anomaly_pixel_indices)
        print(f"len(valid_anomaly_pixel_indices): "
              f"{len(valid_anomaly_pixel_indices)}")  # Anomalies after valid check
        print(f"labels.shape: {labels.shape}")

        if len(valid_anomaly_pixel_indices) > 0:
            for cluster_idx in range(n_clusters):
                # Get indices of pixels belonging to current cluster
                cluster_pixel_indices = valid_anomaly_pixel_indices[
                    labels == cluster_idx
                ]
                if len(cluster_pixel_indices) > 0:
                    # Assign cluster label to corresponding pixels in 2D mask
                    cluster_mask[tuple(cluster_pixel_indices.T)] = (
                        cluster_idx + 1
                    )  # +1 to avoid 0 (background)
                    cluster_color = cluster_cmap(
                        cluster_idx / n_clusters
                    )  # Get color for cluster
                    cluster_patches.append(
                        mpatches.Patch(
                            color=cluster_color,
                            label=f'Cluster {cluster_idx + 1}'
                        )
                    )

    print("-" * 20)
    return cluster_mask, cluster_cmap, cluster_patches, n_clusters



## Visualization

In [16]:
# --- Plotting ---
def plot_results(
    masked_data_list: list,
    cluster_mask_global: np.ndarray,
    cluster_cmap_global: matplotlib.colors.ListedColormap,
    n_clusters_global: int,
    cluster_patches_global: list,
    channel_names: list,
    anomaly_threshold: float,
    output_dir: str
):
    """
    Plots and saves the results, overlaying global clusters on each channel.

    Args:
        masked_data_list (list): List of masked image data arrays.
        cluster_mask_global (np.ndarray): Global cluster mask.
        cluster_cmap_global (matplotlib.colors.ListedColormap): Colormap for clusters.
        n_clusters_global (int): Number of global clusters.
        cluster_patches_global (list): Legend patches for global clusters.
        channel_names (list): List of channel names (wavelengths).
        anomaly_threshold (float): Anomaly threshold used.
        output_dir (str): Directory to save the output figure.
    """
    num_rows, num_cols = 3, 3
    fig, axes = plt.subplots(
        num_rows, num_cols, figsize=(18, 15), dpi=100
    )
    axes = axes.flatten()

    fig.suptitle(
        f'Anomaly Detection in SDO/AIA EUV Channels (K-Means)\n'
        f'Anomaly Threshold: {anomaly_threshold:.2f}',
        fontsize=18
    )

    for i, (masked_data, channel) in enumerate(
        zip(masked_data_list, channel_names)
    ):
        if i < num_rows * num_cols:
            ax = axes[i]

            ax.imshow(
                masked_data,
                origin='lower',
                cmap='YlOrBr',
                vmin=np.nanpercentile(masked_data, 2),
                vmax=np.nanpercentile(masked_data, 98),
                alpha=0.5
            )

            if n_clusters_global > 0:
                for cluster_index in range(1, n_clusters_global + 1):
                    cluster_area_mask = cluster_mask_global == cluster_index
                    cluster_color = cluster_cmap_global(
                        (cluster_index - 1) / n_clusters_global
                    )
                    ax.imshow(
                        np.ma.masked_where(
                            ~cluster_area_mask, cluster_mask_global
                        ),
                        origin='lower',
                        cmap=matplotlib.colors.ListedColormap([cluster_color]),
                        alpha=0.6,
                        vmin=cluster_index - 0.5,
                        vmax=cluster_index + 0.5
                    )

            ax.set_title(
                f'AIA {channel} Å (Clusters: {n_clusters_global})',
                color='black',
                fontsize=14,
                pad=10
            )
            ax.text(
                0.5, -0.18,
                f'Anomaly Threshold: {anomaly_threshold:.2f}',
                ha='center',
                va='center',
                transform=ax.transAxes,
                fontsize=12,
                color='dimgray'
            )
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

            # 👇 Mover la leyenda aquí para que aparezca en cada subplot
            if cluster_patches_global:
                ax.legend(
                    handles=cluster_patches_global,
                    loc='upper right',
                    fontsize='small',
                    frameon=True,
                    handlelength=1,
                    borderpad=0.3
                )

    # Eliminar ejes no usados
    for j in range(len(channel_names), num_rows * num_cols):
        fig.delaxes(axes[j])

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.subplots_adjust(wspace=0.1, hspace=0.3)

    filename = os.path.join(
        output_dir,
        f"kmeans_anomaly_detection_threshold_{anomaly_threshold:.2f}"
        "_global_clusters.png"
    )
    plt.savefig(filename, bbox_inches='tight')
    plt.close(fig)
    print(f"Figure saved to: {filename}")


Test

In [19]:
# Configuración como JSON
config = {
    "data_dir": "sdo_data",
    "channels": ["94", "131", "171", "193", "211", "304", "335", "1600", "1700"],
    "anomaly_thresholds": [0.1],
    "output_dir": "./output_figures",
    "image_size": 512,
    "contamination": 0.05,
    "n_clusters": 7,
    "max_k": 10,
    "random_state": 42
}

def run_pipeline(config):
    os.makedirs(config["output_dir"], exist_ok=True)

    # --- 1. Selección de canales ---
    if config["channels"]:
        channels = [f"aia_{c}" for c in config["channels"]]
    else:
        all_dirs = os.listdir(config["data_dir"])
        channels = [
            d for d in all_dirs
            if os.path.isdir(os.path.join(config["data_dir"], d))
            and not d.startswith("aia_1600")
            and not d.startswith("aia_1700")
        ]

    if not channels:
        print("No se encontraron canales. Terminando.")
        return

    # --- 2. Carga y preprocesamiento ---
    masked_data_list = []
    channel_names = []
    for channel_dir in channels:
        try:
            channel = channel_dir.split("_")[1]
            channel_names.append(channel)
            channel_path = os.path.join(config["data_dir"], channel_dir)
            data, metadata = load_fits_data(channel_path)
            mask = create_circular_mask(data, metadata)
            masked_data = preprocess_image(data, mask, config["image_size"])
            masked_data_list.append(masked_data)
        except Exception as e:
            print(f"Error procesando {channel_dir}: {e}")

    if not masked_data_list:
        print("No se cargaron datos. Terminando.")
        return

    # --- 3. Preparar datos para detección de anomalías ---
    prepared_data, valid_pixel_mask, nan_mask = prepare_data_concatenated(masked_data_list)

    # --- 4. Detección de anomalías ---
    anomaly_scores = detect_anomalies_isolation_forest(prepared_data, config["contamination"])

    anomaly_map_2d = np.full((config["image_size"], config["image_size"]), np.nan)
    anomaly_map_2d[valid_pixel_mask.reshape((config["image_size"], config["image_size"]))] = anomaly_scores

    # --- 5. Umbrales de anomalía ---
    for anomaly_threshold in config["anomaly_thresholds"]:
        print(f"Procesando con umbral de anomalía: {anomaly_threshold}")
        anomaly_mask_global = anomaly_map_2d < anomaly_threshold

        print(f"Anomalías detectadas: {np.sum(anomaly_mask_global)}")

        # --- 6. Clustering ---
        anomaly_pixels_indices = np.argwhere(anomaly_mask_global)
        valid_pixel_mask_2d = ~nan_mask.reshape((config["image_size"], config["image_size"]))
        valid_pixel_indices_2d = np.argwhere(valid_pixel_mask_2d)
        pixel_index_map = {
            tuple(index_2d): i for i, index_2d in enumerate(valid_pixel_indices_2d)
        }

        anomaly_intensity_features = []
        valid_anomaly_pixel_indices = []
        for index_2d in anomaly_pixels_indices:
            if tuple(index_2d) in pixel_index_map:
                row_index = pixel_index_map[tuple(index_2d)]
                anomaly_intensity_features.append(prepared_data[row_index])
                valid_anomaly_pixel_indices.append(index_2d)
        anomaly_intensity_features = np.array(anomaly_intensity_features)
        valid_anomaly_pixel_indices = np.array(valid_anomaly_pixel_indices)

        if len(anomaly_intensity_features) > 0:
            cluster_labels, _ = perform_kmeans_clustering(
                anomaly_intensity_features,
                config["n_clusters"],
                random_state=config["random_state"]
            )

            cluster_mask_global, cluster_cmap_global, cluster_patches_global, \
                n_clusters_global = create_cluster_mask(
                    anomaly_mask_global,
                    cluster_labels,
                    valid_pixel_mask,
                    config["image_size"]
                )
        else:
            print(f"No se encontraron anomalías para el umbral {anomaly_threshold}.")
            cluster_mask_global = np.zeros_like(anomaly_mask_global, dtype=int)
            cluster_cmap_global = matplotlib.colors.ListedColormap([])
            cluster_patches_global = []
            n_clusters_global = 0

        # --- 7. Graficar resultados ---
        plot_results(
            masked_data_list,
            cluster_mask_global,
            cluster_cmap_global,
            n_clusters_global,
            cluster_patches_global,
            channel_names,
            anomaly_threshold,
            config["output_dir"]
        )

# Ejecutar
run_pipeline(config)


Procesando con umbral de anomalía: 0.1
Anomalías detectadas: 32214
--------------------
Inside create_cluster_mask:
anomaly_mask.shape: (512, 512)
np.sum(anomaly_mask): 32214
Number of clusters (n_clusters): 7
len(anomaly_pixels_indices): 32214
len(valid_anomaly_pixel_indices): 32214
labels.shape: (32214,)
--------------------
Figure saved to: ./output_figures\kmeans_anomaly_detection_threshold_0.10_global_clusters.png
